In [1]:
from pathlib import Path
import random
import shutil
import yaml

# ============================================================
# CONFIGURAÇÃO
# ============================================================

RAW_DATA = Path("../raw_data")
DATASET_DIR = Path("dataset_rtdetr_binary")

SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff"
}

random.seed(SEED)

In [2]:
# ============================================================
# 1. ENCONTRAR IMAGENS
# ============================================================

images = [
    p for p in RAW_DATA.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Imagens encontradas:", len(images))


Imagens encontradas: 1170


In [3]:
# ============================================================
# 2. VALIDAR IMAGENS + LABELS
# ============================================================

samples = []

for image in images:
    label = image.with_suffix(".txt")

    if label.exists():
        samples.append(
            (image, label)
        )

print("Imagens com anotação:", len(samples))


Imagens com anotação: 1170


In [4]:
# ============================================================
# 3. CRIAR DATASET
# ============================================================

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val", "test"]:

    (DATASET_DIR / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (DATASET_DIR / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

In [5]:
# ============================================================
# 4. SPLIT
# ============================================================

random.shuffle(samples)

n = len(samples)

n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_samples = samples[:n_train]

val_samples = samples[
    n_train:n_train + n_val
]

test_samples = samples[
    n_train + n_val:
]

print("\nDataset:")

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))


Dataset:
Train: 819
Val: 175
Test: 176


In [6]:
# ============================================================
# 5. CONVERTER LABELS
# ============================================================

def convert_label_to_object(
    source_label,
    destination_label
):

    new_lines = []

    with open(
        source_label,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            # YOLO:
            # class x_center y_center width height

            if len(parts) != 5:
                continue

            # IGNORA a classe original
            #
            # 0 = MILCO
            # 1 = NOMBO
            #
            # Tudo vira:
            # 0 = OBJECT

            _, x, y, w, h = parts

            new_lines.append(
                f"0 {x} {y} {w} {h}\n"
            )

    with open(
        destination_label,
        "w",
        encoding="utf-8"
    ) as f:

        f.writelines(new_lines)



In [7]:
# ============================================================
# 6. COPIAR IMAGENS + CONVERTER LABELS
# ============================================================

def prepare_split(
    samples,
    split
):

    for image, label in samples:
        destination_image = (
            DATASET_DIR
            / "images"
            / split
            / image.name
        )

        destination_label = (
            DATASET_DIR
            / "labels"
            / split
            / label.name
        )

        shutil.copy2(
            image,
            destination_image
        )

        convert_label_to_object(
            label,
            destination_label
        )


prepare_split(train_samples, "train")
prepare_split(val_samples, "val")
prepare_split(test_samples, "test")

In [8]:
# ============================================================
# 7. DATA.YAML
# ============================================================

data_yaml = {
    "path": str(
        DATASET_DIR.resolve()
    ),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "object"
    }
}

yaml_path = DATASET_DIR / "data.yaml"

with open(
    yaml_path,
    "w",
    encoding="utf-8"
) as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


print("\nYAML criado:")
print(yaml_path)


YAML criado:
dataset_rtdetr_binary\data.yaml


In [9]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=1024,
    batch=2,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=5,
    patience=30,

    # --------------------------------
    # AUGMENTATION
    # --------------------------------

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.10,
    degrees=5,
    translate=0.05,
    scale=0.10,
    shear=0,
    perspective=0,
    flipud=0,
    fliplr=0.5,
    mosaic=0.0,
    mixup=0.0,

    # --------------------------------
    # TREINAMENTO
    # --------------------------------

    pretrained=True,
    amp=True,
    deterministic=True,
    seed=42,
    project="runs/rtdetr",
    name="sonar_object_detection",
    save=True,
    plots=True,
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.164 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_rtdetr_binary\data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mod

C:\Users\Usuário\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 23.624.5 MB/s, size: 146.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\dataset_rtdetr_binary\labels\val... 175 images, 134 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 175/175 945.7it/s 0.2ss
val: New cache created: D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\dataset_rtdetr_binary\labels\val.cache
optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0005), 226 bias(decay=0.0)
Plotting labels to D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\runs\detect\runs\rtdetr\sonar_object_detection-2\labels.jpg... 
Usi

C:\Users\Usuário\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\autograd\graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      3.87G      1.245      2.297     0.5532          0       1024: 100% ━━━━━━━━━━━━ 410/410 1.9it/s 3:340.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 1.4it/s 30.6s0.2s
                   all        175         75          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      3.76G          0   0.007514          0          0       1024: 0% ──────────── 0/410  1.2s

...

      48/50      3.92G     0.2256     0.4207    0.04352          0       1024: 100% ━━━━━━━━━━━━ 410/410 2.1it/s 3:110.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 4.7it/s 9.4s0.2s
                   all        175         75      0.231      0.347      0.219      0.107

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      49/50      3.92G          0   0.007847          0          0       1024: 0% ──────────── 0/410  0.6s

C:\Users\Usuário\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\autograd\graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      3.92G     0.2367      0.433    0.04371          3       1024: 100% ━━━━━━━━━━━━ 410/410 2.1it/s 3:120.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 4.7it/s 9.4s0.2s
                   all        175         75      0.277      0.293      0.216      0.105

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


C:\Users\Usuário\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\autograd\graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      3.92G     0.2299     0.4312    0.04268          3       1024: 100% ━━━━━━━━━━━━ 410/410 2.1it/s 3:120.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 4.7it/s 9.3s0.2s
                   all        175         75      0.228      0.293       0.22      0.104

50 epochs completed in 2.844 hours.
Optimizer stripped from D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\runs\detect\runs\rtdetr\sonar_object_detection-2\weights\last.pt, 66.3MB
Optimizer stripped from D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\runs\detect\runs\rtdetr\sonar_object_detection-2\weights\best.pt, 66.3MB

Validating D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\runs\detect\runs\rtdetr\sonar_object_detection-2\weights\best.pt...
Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
rt-detr-l summary (fused): 315 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs
                 Class  

In [12]:
best_model = RTDETR(
    r"D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\runs\detect\runs\rtdetr\sonar_object_detection-2\weights\best.pt"
)

metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=1024,
    batch=2,
    device=0,
    save_json=True,
    plots=True,
    project="runs/rtdetr",
    name="sonar_object_detection_test"
)

Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
rt-detr-l summary (fused): 315 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs
WARNING val: Slow image access detected (ping: 1.22.5 ms, read: 6.05.8 MB/s, size: 152.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\dataset_rtdetr_binary\labels\test... 176 images, 130 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 176/176 103.0it/s 1.7s9s
val: New cache created: D:\testte\SEPARACAO_MODELS\TESTES_COM_RT_DETR\dataset_rtdetr_binary\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 88/88 2.5it/s 34.8s0.2ss0
                   all        176        109      0.247      0.321      0.191     0.0763
Speed: 2.3ms preprocess, 86.9ms inference, 0.0ms loss, 0.5ms postprocess 

In [13]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Precision: 0.24663592484199393
Recall: 0.3211009174311927
mAP50: 0.19108432184971913
mAP50-95: 0.07630530579307254
